# BioAI Hub — Layer 4: NIH ChestX-ray14

En layers anteriores trabajé con el dataset de Kaggle: 5.800 imágenes, clasificación binaria (NORMAL/PNEUMONIA). Ese dataset es útil para aprender el pipeline pero no refleja la complejidad real.

NIH ChestX-ray14 es el dataset de referencia en el área: 112.120 radiografías de 30.805 pacientes distintos, anotadas con 14 patologías por radiólogos. Un paciente puede tener múltiples patologías simultáneamente — eso cambia fundamentalmente cómo se modela el problema.

**Descarga**: ~42 GB. La primera vez tarda 45–90 minutos. Lo guardo directo en Drive para no repetirlo.

## 0. Setup

In [ ]:
import os
import torch
from google.colab import drive

drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/bioai-colab'
NIH_DIR  = os.path.join(WORK_DIR, 'nih-data')   # descarga directa a Drive
os.makedirs(NIH_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"NIH data dir: {NIH_DIR}")

## 1. Descargar NIH ChestX-ray14

Descargo directo a Drive para que persista entre sesiones. La primera vez tarda bastante — es normal.

Si la carpeta `nih-data` ya tiene archivos, la celda no vuelve a descargar.

In [ ]:
# Configurar credenciales de Kaggle
from google.colab import files

if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    print("Subí el kaggle.json...")
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!pip install -q kaggle
print("Kaggle listo")

In [ ]:
csv_path = os.path.join(NIH_DIR, 'Data_Entry_2017.csv')

if os.path.exists(csv_path):
    print("Dataset ya descargado en Drive, salteando...")
else:
    print("Descargando NIH ChestX-ray14 (~42 GB) directo a Drive...")
    print("Esto tarda entre 45 y 90 minutos. No cerrar la sesión.")
    print()
    !kaggle datasets download -d nih-chest-xrays/data -p "{NIH_DIR}" --unzip
    print("\nDescarga completada")

In [ ]:
import glob

archivos = os.listdir(NIH_DIR)
imagenes = glob.glob(os.path.join(NIH_DIR, 'images', '*.png'))

print(f"Archivos en NIH_DIR: {sorted(archivos)}")
print(f"Imágenes encontradas: {len(imagenes):,}")

## 2. Explorar el dataset

El archivo central es `Data_Entry_2017.csv`. Tiene una fila por imagen con las etiquetas separadas por `|`. Por ejemplo: `Atelectasis|Effusion` significa que esa radiografía tiene ambas patologías al mismo tiempo.

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)

print(f"Total de imágenes: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
print()
print(df[['Image Index', 'Finding Labels', 'Patient ID', 'Patient Age', 'Patient Gender']].head(10))

In [ ]:
import matplotlib.pyplot as plt

PATOLOGIAS = [
    'Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax',
    'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia',
    'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]

# Contar prevalencia de cada patología
conteo = {}
for pat in PATOLOGIAS:
    conteo[pat] = df['Finding Labels'].str.contains(pat).sum()

conteo_ordenado = dict(sorted(conteo.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(conteo_ordenado.keys(), conteo_ordenado.values(), color='steelblue', alpha=0.8)
ax.set_title('Prevalencia por patología — NIH ChestX-ray14')
ax.set_ylabel('Número de imágenes')
ax.set_xticklabels(conteo_ordenado.keys(), rotation=40, ha='right')
ax.grid(True, axis='y', alpha=0.3)

for bar, val in zip(bars, conteo_ordenado.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'nih_prevalencia.png'), dpi=120, bbox_inches='tight')
plt.show()

print("\nNo Finding:", df['Finding Labels'].str.contains('No Finding').sum())

In [ ]:
# Cuántas imágenes tienen más de una patología
df['n_patologias'] = df['Finding Labels'].apply(
    lambda x: 0 if x == 'No Finding' else len(x.split('|'))
)

print("Distribución por número de patologías simultáneas:")
print(df['n_patologias'].value_counts().sort_index().to_string())
print()
multi = (df['n_patologias'] > 1).sum()
print(f"Imágenes con 2+ patologías: {multi:,} ({multi/len(df)*100:.1f}%)")

In [ ]:
import matplotlib.image as mpimg

# Mostrar 8 radiografías con sus etiquetas
muestras = df[df['n_patologias'] >= 1].sample(8, random_state=42)

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flatten()

img_dir = os.path.join(NIH_DIR, 'images')

for i, (_, row) in enumerate(muestras.iterrows()):
    img_path = os.path.join(img_dir, row['Image Index'])
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        axes[i].imshow(img, cmap='gray')
        titulo = row['Finding Labels'].replace('|', '\n')
        axes[i].set_title(titulo, fontsize=8)
    axes[i].axis('off')

plt.suptitle('NIH ChestX-ray14 — muestras con etiquetas', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Split train/val/test

El dataset incluye archivos con los splits oficiales del paper original. Es importante usarlos en vez de hacer un split aleatorio — el mismo paciente puede tener múltiples radiografías y si el paciente aparece en train y test el modelo se evalúa sobre datos que ya vio indirectamente.

In [ ]:
def leer_lista(path):
    with open(path) as f:
        return set(line.strip() for line in f if line.strip())

train_val_list = leer_lista(os.path.join(NIH_DIR, 'train_val_list.txt'))
test_list      = leer_lista(os.path.join(NIH_DIR, 'test_list.txt'))

df_train_val = df[df['Image Index'].isin(train_val_list)].copy()
df_test      = df[df['Image Index'].isin(test_list)].copy()

# 90/10 split para train/val dentro del train_val set
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(df_train_val, test_size=0.1, random_state=42)

print(f"Train:  {len(df_train):,} imágenes")
print(f"Val:    {len(df_val):,} imágenes")
print(f"Test:   {len(df_test):,} imágenes")

## 4. Dataset multi-label

En clasificación multi-label cada imagen tiene un vector binario de longitud 14 en vez de un único índice de clase. Por ejemplo:

```
Atelectasis|Effusion  →  [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
No Finding            →  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```

Esto cambia la loss function — CrossEntropyLoss ya no sirve porque asume clases mutuamente excluyentes. En Layer 5 usamos `BCEWithLogitsLoss` con una salida por patología.

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class NIHChestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['Image Index'])
        image    = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        # Vector binario de 14 posiciones
        labels_str = row['Finding Labels']
        label_vec  = torch.zeros(len(PATOLOGIAS), dtype=torch.float32)
        if labels_str != 'No Finding':
            for pat in labels_str.split('|'):
                if pat in PATOLOGIAS:
                    label_vec[PATOLOGIAS.index(pat)] = 1.0

        return image, label_vec

print("Clase NIHChestDataset definida")

In [ ]:
from torch.utils.data import DataLoader

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

img_dir = os.path.join(NIH_DIR, 'images')

train_ds = NIHChestDataset(df_train, img_dir, transform=transform_train)
val_ds   = NIHChestDataset(df_val,   img_dir, transform=transform_eval)
test_ds  = NIHChestDataset(df_test,  img_dir, transform=transform_eval)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader:   {len(val_loader)} batches")
print(f"Test loader:  {len(test_loader)} batches")

In [ ]:
# Verificar que un batch tiene la forma correcta
imagenes, etiquetas = next(iter(train_loader))

print(f"Shape imágenes: {imagenes.shape}")
print(f"Shape etiquetas: {etiquetas.shape}")
print()

# Mostrar etiquetas del primer elemento del batch
labels_primer = etiquetas[0]
patologias_presentes = [PATOLOGIAS[i] for i, v in enumerate(labels_primer) if v == 1]
print(f"Primera imagen — patologías presentes: {patologias_presentes if patologias_presentes else ['No Finding']}")
print(f"Vector binario: {labels_primer.numpy().astype(int).tolist()}")

In [ ]:
# Pesos para BCEWithLogitsLoss — cada patología tiene su propio peso
# porque la prevalencia varía mucho entre clases
total = len(df_train)
pos_weights = []

print("Prevalencia en training set:")
for pat in PATOLOGIAS:
    n_pos = df_train['Finding Labels'].str.contains(pat).sum()
    n_neg = total - n_pos
    peso  = n_neg / (n_pos + 1e-6)   # ratio neg/pos
    pos_weights.append(peso)
    print(f"  {pat:25s}  {n_pos:5,} casos  peso={peso:.1f}")

pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
print(f"\nTensor de pesos shape: {pos_weights_tensor.shape}")

## 5. Qué sigue

El DataLoader está listo y los pesos calculados. En Layer 5 uso este mismo loader para hacer fine-tuning de EfficientNet-B3 con `BCEWithLogitsLoss` — una salida por patología en vez de softmax sobre clases mutuamente excluyentes.

In [ ]:
# Guardar los splits y pesos en Drive para no recalcular en Layer 5
import pickle

estado = {
    'df_train': df_train,
    'df_val':   df_val,
    'df_test':  df_test,
    'pos_weights': pos_weights,
    'patologias': PATOLOGIAS,
}

with open(os.path.join(WORK_DIR, 'nih_splits.pkl'), 'wb') as f:
    pickle.dump(estado, f)

print("Estado guardado en Drive")
print()
print("Archivos en Drive:")
for archivo in sorted(os.listdir(WORK_DIR)):
    size = os.path.getsize(os.path.join(WORK_DIR, archivo))
    if os.path.isfile(os.path.join(WORK_DIR, archivo)):
        print(f"  {archivo:45s} {size/1e6:.1f} MB")
    else:
        print(f"  {archivo}/ (carpeta)")